# Metadata Panel and Topic Modeling Workflow

This notebook is the working view for the metadata export. It audits the available XML/CSV fields, builds the document-level metadata panel, and then reads the prepared outputs in a report-friendly order.

## Setup

The notebook keeps the current TDM Studio Python kernel and uses repository-relative paths so it can be rerun from the project root or from the `analysis/` folder.

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import Image, display, Markdown

ROOT = Path.cwd()
if not (ROOT / 'scripts').exists() and (ROOT.parent / 'scripts').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'scripts'))

from eda_helpers import (
    scan_xml_field_inventory,
    write_xml_field_manifest,
    build_metadata_panel_outputs,
    write_metadata_export_manifest,
)

OUT = ROOT / 'analysis' / 'outputs'
FINAL = ROOT / 'analysis' / 'final'
FIG = OUT / 'figures'
pd.set_option('display.max_colwidth', 120)
ROOT

## XML and CSV Field Availability

The TDM metadata README says columns are omitted when no document has a given tag. This audit therefore separates confirmed CSV fields from XML candidate paths that should be scanned when XML exports are available.

In [ ]:
write_xml_field_manifest()

In [ ]:
scan_xml_field_inventory(max_files=100)

## Build the Metadata Panel

This is the efficient export step. It reuses the cleaned parquet when available, adds journal/publication ranges and coverage density, and creates both title-only and metadata-enriched text fields for topic-modeling experiments.

In [ ]:
build_metadata_panel_outputs()

In [ ]:
write_metadata_export_manifest()

In [ ]:
pd.read_json(FINAL / 'metadata_panel_summary.json', typ='series')

In [ ]:
pd.read_csv(FINAL / 'metadata_export_manifest.csv')

## Panel Field Definitions

In [ ]:
pd.read_csv(FINAL / 'metadata_panel_data_dictionary.csv')

## Journal Range and Coverage

In [ ]:
pd.read_csv(OUT / 'journal_coverage_summary.csv').head(25)

## Metadata Panel Preview

In [ ]:
panel_preview_cols = [
    'goid', 'title', 'date', 'source_type', 'object_type',
    'publication_title_display', 'journal_range_start', 'journal_range_end',
    'full_text_status', 'primary_author', 'metadata_suggests_full_text_record'
]
pd.read_parquet(FINAL / 'metadata_panel.parquet', columns=panel_preview_cols).head(20)

## Core Summary

In [ ]:
pd.read_csv(OUT / 'core_summary.csv')

In [ ]:
display(Image(filename=str(FIG / 'core_coverage_snapshot.png')))

## Metadata Field Audit

In [ ]:
pd.read_csv(FINAL / 'metadata_field_audit.csv')

## Final Cleaned Dataset

In [ ]:
pd.read_json(FINAL / 'cleaned_metadata_summary.json', typ='series')

In [ ]:
pd.read_csv(FINAL / 'cleaned_metadata_data_dictionary.csv')

## Scope

In [ ]:
pd.read_csv(OUT / 'publisher_region_counts.csv')

In [ ]:
pd.read_csv(OUT / 'edition_flag_counts.csv')

In [ ]:
pd.read_csv(OUT / 'excluded_media_summary.csv')

## Media Coverage

In [ ]:
pd.read_csv(OUT / 'media_summary.csv').head(20)

In [ ]:
display(Image(filename=str(FIG / 'media_summary_counts.png')))

## Time Coverage

In [ ]:
pd.read_csv(OUT / 'decade_counts.csv')

In [ ]:
display(Image(filename=str(FIG / 'documents_by_year.png')))

## Source Type Over Time

In [ ]:
pd.read_csv(OUT / 'source_type_by_year_english_na.csv').head(20)

In [ ]:
display(Image(filename=str(FIG / 'source_type_by_year_lines.png')))

In [ ]:
display(Image(filename=str(FIG / 'source_type_by_year_since_1980.png')))

## Issue Coverage

In [ ]:
pd.read_csv(OUT / 'core_issue_coverage.csv')

In [ ]:
pd.read_csv(OUT / 'top_subject_terms.csv').head(20)

In [ ]:
pd.read_csv(OUT / 'top_class_terms.csv').head(15)

## Data Quality

In [ ]:
pd.read_csv(OUT / 'duplicate_summary.csv')

In [ ]:
pd.read_csv(OUT / 'column_completeness.csv')

In [ ]:
pd.read_csv(OUT / 'publisher_city_counts.csv').head(20)

In [ ]:
pd.read_csv(OUT / 'publisher_city_raw_counts.csv').head(20)

## Topic Modeling Inputs

The existing model remains title-only for conservative interpretation. The panel also exposes `metadata_enriched_topic_text` so subject/class-term sensitivity runs can be compared without changing the unit of observation.

In [ ]:
pd.read_json(OUT / 'topic_model_profile.json', typ='series')

In [ ]:
pd.read_csv(OUT / 'topic_counts.csv')

In [ ]:
pd.read_csv(OUT / 'topic_terms.csv').query('Rank <= 8')

In [ ]:
display(Image(filename=str(FIG / 'topic_counts.png')))

In [ ]:
display(Image(filename=str(FIG / 'topic_by_decade_heatmap.png')))

In [ ]:
display(Image(filename=str(FIG / 'topic_by_media_heatmap.png')))

## Final Files

In [ ]:
pd.DataFrame({'file': [p.name for p in sorted(FINAL.glob('*'))], 'bytes': [p.stat().st_size for p in sorted(FINAL.glob('*'))]})